# 0장 — 오리엔테이션 실습

교재 `docs/book/00-orientation.md` 와 함께 본다. 이 노트북에서 확인하는 것:

1. PyTorch 가 CPU 를 어떻게 쓰는지 (스레드 수, 행렬곱 속도)
2. 코퍼스의 크기와 글자 분포
3. 텐서 shape 감각 — `(B, T, C)`

> 셀은 위에서 아래로 `Shift+Enter` 로 실행한다. 자바의 `main()` 하나가 아니라, 셀마다 상태가 이어지는 REPL 이다.

## 1. 환경 확인

In [ ]:
import time
import torch

from shllm.config import DATA_DIR, setup_cpu

device = setup_cpu()  # 스레드 수를 코어 수로, 시드 고정
print("torch", torch.__version__)
print("device", device, "| threads", torch.get_num_threads())
print("data dir", DATA_DIR, "->", DATA_DIR.resolve())

행렬곱 한 번의 속도를 재 본다. LLM 의 계산은 거의 전부 행렬곱이라, 이 숫자가 곧 "이 PC 의 학습 속도"다.
GFLOPS(초당 10억 번 곱셈-덧셈)로 환산하면 GPU 와의 차이를 감으로 잡을 수 있다 (소비자용 GPU 는 수천~수만).

In [ ]:
n = 2048
a = torch.randn(n, n)
b = torch.randn(n, n)
t0 = time.perf_counter()
c = a @ b
dt = time.perf_counter() - t0
print(f"{n}x{n} 행렬곱: {dt*1000:.0f} ms  →  {2*n**3/dt/1e9:.1f} GFLOPS")

## 2. 코퍼스 살펴보기

In [ ]:
from collections import Counter

from shllm.data import list_work_files, load_corpus

works = list_work_files()
print(len(works), "편:", ", ".join(p.stem for p in works[:5]), "...")
text = load_corpus("korean-classics")
print(f"합본: 총 {len(text):,} 자, 고유 글자 {len(set(text)):,} 종")
print("---")
print(text[:300])

In [ ]:
counts = Counter(text)
top = counts.most_common(20)
for ch, n in top:
    print(repr(ch), n)

고유 글자가 수천 종이다 — 한글 음절만 해도 11,172자가 가능하고 한자·기호까지 섞여 있다.
영어(알파벳 26자 + 기호 ≈ 65종)와 비교하면 **"문자 하나 = 토큰 하나"가 한글에서 왜 비효율적인지**가 여기서 드러난다. 1~2장에서 이 문제를 다룬다.

## 3. 텐서 shape 감각

LLM 코드에서는 텐서 세 개의 차원을 항상 `(B, T, C)` 로 부른다.

- **B** batch — 한 번에 처리하는 문장 수
- **T** time — 문장 안의 토큰 수 (문맥 길이)
- **C** channels — 토큰 하나를 나타내는 숫자 개수 (임베딩 차원)

자바로 치면 `double[B][T][C]` 인데, 연산이 루프 없이 한 번에 적용된다는 점만 다르다.

In [ ]:
B, T, C = 4, 8, 16
x = torch.randn(B, T, C)
print("x", tuple(x.shape))

# 각 토큰 벡터의 평균 (C 차원을 접는다)
print("x.mean(dim=-1)", tuple(x.mean(dim=-1).shape))        # (B, T)

# 선형 변환: C -> 32.  W 는 (C, 32) 이고 마지막 차원끼리 곱해진다
W = torch.randn(C, 32)
y = x @ W
print("x @ W", tuple(y.shape))                              # (B, T, 32)

# 토큰끼리의 유사도: (B, T, C) x (B, C, T) -> (B, T, T)  ← 5장 어텐션의 핵심 모양
sim = x @ x.transpose(-2, -1)
print("x @ x^T", tuple(sim.shape))

`(B, T, T)` 모양이 나왔다. "문장 안의 모든 토큰 쌍에 대해 숫자 하나" — 5장의 어텐션 행렬이 정확히 이 모양이다.
지금은 모양만 눈에 익혀 둔다.

---
**다음 장**: 1장 — 글자를 숫자로 바꾸고, 확률표만으로 한글을 생성해 본다.